In [1]:
!pip install -q gradio

In [2]:
import gradio as gr
import re

In [3]:
def detect_bugs(code, language):
    bugs = []

    # Empty code
    if not code.strip():
        return [{
            "type": "Input Error",
            "severity": "HIGH",
            "message": "No source code was provided."
        }]

    # Python checks
    if language == "Python":

        if "input(" in code and "try:" not in code:
            bugs.append({
                "type": "Input Handling",
                "severity": "MEDIUM",
                "message": "User input is not protected against invalid input."
            })

        if "/ 0" in code or "/0" in code:
            bugs.append({
                "type": "Division by Zero",
                "severity": "HIGH",
                "message": "The code may attempt to divide by zero."
            })

        if "eval(" in code:
            bugs.append({
                "type": "Unsafe Evaluation",
                "severity": "HIGH",
                "message": "eval() can execute dynamically supplied code."
            })

        if "password =" in code.lower():
            bugs.append({
                "type": "Hardcoded Credential",
                "severity": "HIGH",
                "message": "A password appears to be hardcoded in the source code."
            })

        if "except:" in code:
            bugs.append({
                "type": "Exception Handling",
                "severity": "MEDIUM",
                "message": "A bare except statement can hide unexpected errors."
            })

        if "TODO" in code:
            bugs.append({
                "type": "Incomplete Code",
                "severity": "LOW",
                "message": "The source contains a TODO item that may require implementation."
            })

    # Java checks
    elif language == "Java":

        if "System.out.println" not in code:
            bugs.append({
                "type": "Output Check",
                "severity": "LOW",
                "message": "No console output was detected."
            })

        if "catch (Exception" in code:
            bugs.append({
                "type": "Exception Handling",
                "severity": "MEDIUM",
                "message": "Generic Exception handling may hide specific errors."
            })

        if "password =" in code.lower():
            bugs.append({
                "type": "Hardcoded Credential",
                "severity": "HIGH",
                "message": "A password appears to be hardcoded."
            })

    # C/C++ checks
    elif language in ["C", "C++"]:

        if "gets(" in code:
            bugs.append({
                "type": "Unsafe Input",
                "severity": "HIGH",
                "message": "gets() is unsafe because it can cause buffer overflow."
            })

        if "scanf(" in code and "&" not in code:
            bugs.append({
                "type": "Input Handling",
                "severity": "MEDIUM",
                "message": "Check scanf() arguments and ensure variables are passed correctly."
            })

        if "TODO" in code:
            bugs.append({
                "type": "Incomplete Code",
                "severity": "LOW",
                "message": "The source contains a TODO item."
            })

    return bugs

In [4]:
def generate_suggestions(code, language, bugs):

    suggestions = []

    if not bugs:
        suggestions.append(
            "No major issues were detected. Continue following clean-code practices."
        )

    for bug in bugs:

        if bug["type"] == "Division by Zero":
            suggestions.append(
                "Check the denominator before performing division."
            )

        elif bug["type"] == "Hardcoded Credential":
            suggestions.append(
                "Store credentials securely using environment variables or a secret manager."
            )

        elif bug["type"] == "Unsafe Evaluation":
            suggestions.append(
                "Avoid eval() and use safer alternatives for processing input."
            )

        elif bug["type"] == "Input Handling":
            suggestions.append(
                "Validate user input and handle invalid values using appropriate exception handling."
            )

        elif bug["type"] == "Exception Handling":
            suggestions.append(
                "Catch specific exceptions instead of using a generic exception handler."
            )

        elif bug["type"] == "Unsafe Input":
            suggestions.append(
                "Replace unsafe input functions with safer alternatives such as fgets()."
            )

        elif bug["type"] == "Incomplete Code":
            suggestions.append(
                "Review and complete the TODO section before deployment."
            )

    # General suggestions
    if len(code.splitlines()) > 30:
        suggestions.append(
            "Consider splitting large functions or modules into smaller reusable components."
        )

    if language == "Python":
        suggestions.append(
            "Follow PEP 8 naming, indentation, and formatting conventions."
        )

    elif language == "Java":
        suggestions.append(
            "Use meaningful variable names and keep classes and methods focused."
        )

    return list(dict.fromkeys(suggestions))

In [5]:
def calculate_score(bugs):

    score = 100

    for bug in bugs:
        if bug["severity"] == "HIGH":
            score -= 25
        elif bug["severity"] == "MEDIUM":
            score -= 15
        elif bug["severity"] == "LOW":
            score -= 5

    score = max(0, score)

    if score >= 80:
        status = "GOOD"
    elif score >= 60:
        status = "MEDIUM RISK"
    else:
        status = "HIGH RISK"

    return score, status

In [6]:
def review_code(code, language):

    bugs = detect_bugs(code, language)

    suggestions = generate_suggestions(
        code,
        language,
        bugs
    )

    score, status = calculate_score(bugs)

    # Bug report
    if bugs:
        bug_report = ""

        for i, bug in enumerate(bugs, 1):
            bug_report += (
                f"### Bug {i}\n"
                f"**Type:** {bug['type']}\n\n"
                f"**Severity:** {bug['severity']}\n\n"
                f"**Explanation:** {bug['message']}\n\n"
                "---\n\n"
            )
    else:
        bug_report = "### No major bugs detected ✅"

    # Suggestions
    suggestion_report = ""

    for i, suggestion in enumerate(suggestions, 1):
        suggestion_report += f"{i}. {suggestion}\n\n"

    # Summary
    summary = f"""
## Code Review Summary

**Programming Language:** {language}

**Bugs Detected:** {len(bugs)}

**Code Quality Score:** {score}/100

**Risk Level:** {status}
"""

    return summary, bug_report, suggestion_report

In [7]:
with gr.Blocks(title="CodeClarity AI - Code Review Interface") as demo:

    gr.Markdown(
        """
        # 🔍 CodeClarity AI
        ## Intelligent Code-Review Interface

        Paste your source code below and get an automated review
        containing bug detection, severity, explanations and
        improvement suggestions.
        """
    )

    with gr.Row():

        with gr.Column():

            language = gr.Dropdown(
                choices=[
                    "Python",
                    "Java",
                    "C",
                    "C++"
                ],
                value="Python",
                label="Programming Language"
            )

            code_input = gr.Code(
                label="Enter Your Source Code",
                language="python",
                lines=20
            )

            review_button = gr.Button(
                "🔎 Review Code",
                variant="primary"
            )

        with gr.Column():

            summary_output = gr.Markdown(
                label="Review Summary"
            )

            bug_output = gr.Markdown(
                label="Bug Detection"
            )

            suggestion_output = gr.Markdown(
                label="Improvement Suggestions"
            )

    review_button.click(
        fn=review_code,
        inputs=[
            code_input,
            language
        ],
        outputs=[
            summary_output,
            bug_output,
            suggestion_output
        ]
    )

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1e327c2c2a756db31d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
